In [1]:
import sys
!{sys.executable} -m pip install langchain langchain-community langchain-text-splitters langchain-chroma chromadb langchain-ollama pypdf beautifulsoup4

In [2]:
from langchain_community.document_loaders import PyPDFLoader
loader=PyPDFLoader('/Users/parikhandelwal/Desktop/folder/Rag/PDF_Chatbot_RAG_Learning_Roadmap.pdf')
docs=loader.load()
print(docs)
print(len(docs))

/var/folders/9j/zl004qg95nj7r8zmgmqqd4bm0000gn/T/ipykernel_4613/14988773.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-06-24T09:16:55+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-06-24T09:16:55+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '/Users/parikhandelwal/Desktop/folder/Rag/PDF_Chatbot_RAG_Learning_Roadmap.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='PDF Chatbot (RAG) Learning Roadmap\nThis document outlines the concepts, tools, architecture, and learning path required to build a PDF\nChatbot using Retrieval-Augmented Generation (RAG).\n1. PDF Reading\nUse pypdf to extract text from PDF documents. Goal: Convert PDF files into plain text.\n2. Text Chunking\nSplit large documents into smaller chunks. Learn chunk size, overlap, recursive chunking, and\nsemantic chunking.\n3. Embeddings\nConvert chunks into vectors using sentence-transformers such as all-MiniLM-L6-v2.\n4. Vector Storage\nSto

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=50)
splits=text_splitter.split_documents(docs)
print(splits[0])
print(len(splits))

page_content='PDF Chatbot (RAG) Learning Roadmap
This document outlines the concepts, tools, architecture, and learning path required to build a PDF
Chatbot using Retrieval-Augmented Generation (RAG).
1. PDF Reading
Use pypdf to extract text from PDF documents. Goal: Convert PDF files into plain text.
2. Text Chunking
Split large documents into smaller chunks. Learn chunk size, overlap, recursive chunking, and
semantic chunking.
3. Embeddings
Convert chunks into vectors using sentence-transformers such as all-MiniLM-L6-v2.
4. Vector Storage
Store embeddings in FAISS. Later explore Pinecone and Weaviate.
5. Retrieval
Convert user questions into embeddings and retrieve the most relevant chunks using similarity
search.
6. Generation
Provide retrieved chunks and user questions to an LLM such as OpenAI, Gemini, or Llama to
generate answers.' metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-06-24T09:16:55+00:00', 'author': '(anon

In [4]:
from langchain_ollama import OllamaEmbeddings
embedding = OllamaEmbeddings(model="bge-m3")

In [5]:
from langchain_chroma import Chroma
vectorstore=Chroma.from_documents(documents=splits,embedding=embedding)

In [6]:
print(vectorstore._collection.count())

2


In [7]:
print(vectorstore._collection.get())

{'ids': ['a95c01dc-f0d8-44a3-b105-9e380d9bb0bc', 'cb1a5435-10bb-4aa9-a8ae-a17837cc9ea1'], 'embeddings': None, 'documents': ['PDF Chatbot (RAG) Learning Roadmap\nThis document outlines the concepts, tools, architecture, and learning path required to build a PDF\nChatbot using Retrieval-Augmented Generation (RAG).\n1. PDF Reading\nUse pypdf to extract text from PDF documents. Goal: Convert PDF files into plain text.\n2. Text Chunking\nSplit large documents into smaller chunks. Learn chunk size, overlap, recursive chunking, and\nsemantic chunking.\n3. Embeddings\nConvert chunks into vectors using sentence-transformers such as all-MiniLM-L6-v2.\n4. Vector Storage\nStore embeddings in FAISS. Later explore Pinecone and Weaviate.\n5. Retrieval\nConvert user questions into embeddings and retrieve the most relevant chunks using similarity\nsearch.\n6. Generation\nProvide retrieved chunks and user questions to an LLM such as OpenAI, Gemini, or Llama to\ngenerate answers.', 'RAG Architecture\nPDF

In [8]:
print(vectorstore._collection.get(ids=['71c579c2-cd98-4df9-92cd-a4861e27169f'],include=['metadatas','embeddings','documents']))

{'ids': [], 'embeddings': array([], dtype=float64), 'documents': [], 'uris': None, 'included': ['metadatas', 'embeddings', 'documents'], 'data': None, 'metadatas': []}


In [9]:
retriever=vectorstore.as_retriever()

In [10]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are an intelligent AI assistant specialized in answering questions using a PDF document.

Your primary goal is to answer the user's question accurately and clearly.

Instructions:

1. Always use the provided context as the primary source of information.

2. If the answer is completely available in the context, answer only using the context.

3. If the context is incomplete or does not contain enough information, use your own knowledge to complete the answer.

4. Whenever you use knowledge outside the provided context, clearly mention:

Note:
The following information is based on my general knowledge because the provided PDF does not contain enough information.

5. Write answers in a professional, well-structured format.

6. Use headings and bullet points whenever appropriate.

7. If useful, include an example.

8. At the end of every answer, recommend 3–5 closely related topics.

Display them exactly like this:

Related Topics:
- Topic 1
- Topic 2
- Topic 3
- Topic 4
- Topic 5

Do not ask the user any questions.
Do not ask for Yes/No.
Do not ask the user to continue.
Only provide the answer and related topics.

Context:
{context}

Question:
{question}

Answer:
""")

In [11]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="mistral",
    temperature=0
)

In [12]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [13]:

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [14]:
rag_chain=({"context":retriever| format_docs,"question":RunnablePassthrough()}|prompt|llm|StrOutputParser())

In [15]:
import re
def valid_question(question):
    question = question.strip()
    # Empty input
    if question == "":
        return False
    # Greetings
    greetings = {
        "hi",
        "hello",
        "hey",
        "good morning",
        "good afternoon",
        "good evening"
    }
    if question.lower() in greetings:
        return False
    # Single character
    if len(question) == 1:
        return False
    # Numbers only
    if question.isdigit():
        return False
    # Symbols only
    if re.fullmatch(r"[^\w]+", question):
        return False
    return True
while True:
    question = input("\nAsk your question (or type 'exit' to quit): ").strip()
    # Exit chatbot
    if question.lower() in ["exit", "quit"]:
        print("\nThank you for using this AI assistant. Have a wonderful day!")
        break
    # Validate question
    if not valid_question(question):
        print("\nSorry! Invalid question. Please enter a complete and meaningful question.")
        continue
    # Get answer from RAG
    response = rag_chain.invoke(question)
    print("\nAnswer:\n")
    print(response)
    # Ask if the user wants to learn more
    while True:
        choice = input("\nWould you like to know more? (Yes/No): ").strip().lower()
        if choice in ["yes", "y"]:
            topic = input("\nEnter the related topic you want to learn about: ").strip()
            if not valid_question(topic):
                print("\nPlease enter a valid topic.")
                continue
            response = rag_chain.invoke(f"Explain in detail about {topic}")
            print("\nAnswer:\n")
            print(response)
        elif choice in ["no", "n"]:
            print("\nOkay! Ask me another question.")
            break
        else:
            print("\nPlease type only Yes/Y or No/N.")


Ask your question (or type 'exit' to quit):  summarize the pdf



Answer:

 The provided document outlines a learning roadmap for building a PDF Chatbot using Retrieval-Augmented Generation (RAG) architecture. The goal is to create a complete PDF chatbot from scratch before utilizing frameworks.

The RAG architecture consists of the following steps:
1. PDF Reading: Extract text from PDF documents using pypdf library.
2. Text Chunking: Split large documents into smaller chunks, learning about chunk size, overlap, recursive chunking, and semantic chunking.
3. Embeddings: Convert chunks into vectors using sentence-transformers such as all-MiniLM-L6-v2.
4. Vector Storage: Store embeddings in FAISS. Later explore Pinecone and Weaviate.
5. Retrieval: Convert user questions into embeddings and retrieve the most relevant chunks using similarity search.
6. Generation: Provide retrieved chunks and user questions to an LLM such as OpenAI, Gemini, or Llama to generate answers.

Related Topics:
- PDF Reading (Extraction)
- Text Chunking Techniques
- Embeddings f


Would you like to know more? (Yes/No):  no



Okay! Ask me another question.



Ask your question (or type 'exit' to quit):  quit



Thank you for using this AI assistant. Have a wonderful day!
